# Creative Pitch Pipeline (Minimal UI)

This notebook only exposes direct prototype controls and stage toggles.

- Stage order: `image_gen` -> `animation_gen` -> `image_extract` -> `upscale`
- Direct config controls: `fallback_frame_count`, `scene_limit`, `webp_quality`
- Image-gen controls: `generate_start_frame`, `start_frame_variants`, `generate_end_frame`
- Pipeline logic lives in Python files, not notebook cells.


In [1]:
from pathlib import Path
import importlib
import sys

import ipywidgets as widgets
from IPython.display import display

start = Path.cwd().resolve()
REPO_ROOT = next((candidate for candidate in [start, *start.parents] if (candidate / "package.json").exists()), start)
PIPELINE_DIR = (REPO_ROOT / "creative-pitch" / "pipeline").resolve()
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

import pipeline_backend
pipeline_backend = importlib.reload(pipeline_backend)
STAGES = pipeline_backend.STAGES
load_config = pipeline_backend.load_config
run_pipeline = pipeline_backend.run_pipeline

config = load_config(REPO_ROOT)
print(f"Loaded config from: {PIPELINE_DIR / 'config.json'}")
print("Stage order:", " -> ".join(stage.key for stage in STAGES))

commands = config.get("commands", {})
missing = [stage.key for stage in STAGES if not stage.default_command and not commands.get(stage.key)]
if missing:
    print("Missing commands (configure in config.json):", ", ".join(missing))


Loaded config from: /Users/ben/Documents/chaskipitch/creative-pitch/pipeline/config.json
Stage order: image_gen -> animation_gen -> image_extract -> upscale


In [2]:
defaults = config.get("defaults", {})
render = config.get("render", {})

story_scene_count = 0
story_path = REPO_ROOT / "creative-pitch" / "story.json"
try:
    import json
    with story_path.open("r", encoding="utf-8") as handle:
        story_payload = json.load(handle)
    if isinstance(story_payload, dict) and isinstance(story_payload.get("scenes"), list):
        story_scene_count = len(story_payload["scenes"])
except Exception:
    story_scene_count = 0

scene_limit_max = max(1, story_scene_count or 50)

default_frame_count = widgets.IntSlider(
    value=int(defaults.get("frame_count", 24)),
    min=8,
    max=60,
    step=1,
    description="Frames*"
)
scene_limit = widgets.IntSlider(
    value=max(0, min(int(defaults.get("scene_limit", 0)), scene_limit_max)),
    min=0,
    max=scene_limit_max,
    step=1,
    description="Scenes"
)
webp_quality = widgets.IntSlider(
    value=int(render.get("webp_quality", 82)),
    min=40,
    max=100,
    step=1,
    description="WebP Q"
)

generate_start_frame = widgets.Checkbox(
    value=bool(defaults.get("generate_start_frame", True)),
    description="Gen Start",
    indent=False,
)
start_frame_variants = widgets.IntSlider(
    value=max(1, min(12, int(defaults.get("start_frame_variants", 4)))),
    min=1,
    max=12,
    step=1,
    description="Start Vars",
)
generate_end_frame = widgets.Checkbox(
    value=bool(defaults.get("generate_end_frame", False)),
    description="Gen End",
    indent=False,
)

frame_help = widgets.HTML("<small>*Used only as fallback when a scene has no generated video frames yet.</small>")
scene_help = widgets.HTML(f"<small>0 = all scenes (story scenes: {story_scene_count})</small>")
image_help = widgets.HTML(
    "<small>Start workflow: generate start options -> pick one as <code>start_selected.png</code> in <code>creative-pitch/pipeline/images/&lt;scene&gt;/&lt;sequence&gt;/</code> -> run end generation.</small>"
)

stage_checkboxes = {
    stage.key: widgets.Checkbox(value=True, description=stage.label, indent=False)
    for stage in STAGES
}

prototype_btn = widgets.Button(description="Apply Prototype Preset", button_style="warning")
run_btn = widgets.Button(description="Run Selected Stages", button_style="success")
output = widgets.Output(layout=widgets.Layout(border="1px solid #ccc", max_height="500px", overflow="auto"))


def apply_prototype_preset(_):
    default_frame_count.value = 12
    scene_limit.value = min(3, scene_limit_max)
    webp_quality.value = 72
    generate_start_frame.value = True
    start_frame_variants.value = 4
    generate_end_frame.value = False


def _update_start_variant_state(_=None):
    start_frame_variants.disabled = not bool(generate_start_frame.value)


def logger(message: str):
    with output:
        print(message)


def on_run(_):
    output.clear_output()
    selected = [key for key, checkbox in stage_checkboxes.items() if checkbox.value]
    if not selected:
        with output:
            print("No stages selected.")
        return

    run_config = {
        "defaults": {
            "frame_count": int(default_frame_count.value),
            "scene_limit": int(scene_limit.value),
            "generate_start_frame": bool(generate_start_frame.value),
            "start_frame_variants": int(start_frame_variants.value),
            "generate_end_frame": bool(generate_end_frame.value),
        },
        "render": {
            "webp_quality": int(webp_quality.value)
        }
    }

    result = run_pipeline(selected, config=run_config, repo_root=REPO_ROOT, logger=logger)
    with output:
        print(f"\nDone. status={result.get('status')}")


prototype_btn.on_click(apply_prototype_preset)
run_btn.on_click(on_run)
generate_start_frame.observe(_update_start_variant_state, names="value")
_update_start_variant_state()

controls = widgets.VBox([
    widgets.HTML("<h3>Stage Toggles</h3>"),
    widgets.VBox(list(stage_checkboxes.values())),
    widgets.HTML("<h3>Direct Config</h3>"),
    default_frame_count,
    frame_help,
    scene_limit,
    scene_help,
    webp_quality,
    widgets.HTML("<h3>Image Gen Controls</h3>"),
    widgets.HBox([generate_start_frame, start_frame_variants, generate_end_frame]),
    image_help,
    widgets.HBox([prototype_btn, run_btn])
])

display(controls)
display(output)


Output(layout=Layout(border_bottom='1px solid #ccc', border_left='1px solid #ccc', border_right='1px solid #cc…